# 📈 Hệ thống dự báo rời bỏ dịch vụ (Customer Churn Prediction)

**Tác giả:** Nguyễn Trọng Nam  
**Mục tiêu:** Xây dựng quy trình Machine Learning hoàn chỉnh để dự đoán khả năng rời bỏ dịch vụ của khách hàng viễn thông dựa trên tập dữ liệu IBM Telco Customer Churn.

## Quy trình thực hiện:
1. **Tiền xử lý dữ liệu:** Làm sạch, xử lý giá trị thiếu và chuẩn hóa đặc trưng.
2. **Phân tích khám phá (EDA):** Tìm kiếm các quy luật và mối liên hệ giữa các đặc trưng và biến mục tiêu.
3. **Huấn luyện mô hình:** So sánh 3 thuật toán phổ biến: Logistic Regression, Decision Tree và Random Forest.
4. **Đánh giá & Triển khai:** Lựa chọn mô hình tối ưu và kiểm thử khả năng dự đoán.

## 1. Cài đặt và import thư viện

Sử dụng các thư viện chuẩn như `pandas`, `scikit-learn` và các module được đóng gói trong thư mục `src/` để đảm bảo tính tái sử dụng cao.

In [1]:
import sys
from pathlib import Path

# Thiết lập đường dẫn để import các module từ thư mục src
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import RAW_DATA_PATH, RANDOM_STATE, TEST_SIZE
from src.data_processing import clean_telco_data, load_telco_data, split_features_target
from src.modeling import train_and_evaluate_models
from src.visualization import (
    plot_target_distribution,
    plot_churn_by_category,
    plot_numeric_by_churn,
    plot_model_metrics,
    plot_confusion_matrix,
)


## 2. Tải và làm sạch dữ liệu

Dữ liệu gốc chứa 7,043 bản ghi. Chúng ta cần xử lý cột `TotalCharges` (chứa các ô trống) và chuẩn hóa biến `SeniorCitizen`.

In [2]:
raw_df = load_telco_data(RAW_DATA_PATH)
df = clean_telco_data(raw_df)

print('Kích thước dữ liệu sau làm sạch:', df.shape)
display(df.head())


## 3. Phân tích khám phá dữ liệu (EDA)

Giai đoạn này giúp ta hiểu rõ các yếu tố nào ảnh hưởng mạnh nhất đến quyết định rời bỏ của khách hàng.

In [3]:
plot_target_distribution(df)
for col in ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']:
    plot_churn_by_category(df, col)
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    plot_numeric_by_churn(df, col)


### 💡 Các phát hiện quan trọng từ EDA:
1. **Loại hợp đồng (Contract):** Khách hàng sử dụng hợp đồng **Month-to-month** có tỷ lệ rời bỏ cao hơn rõ rệt so với hợp đồng dài hạn (One year/Two year). Đây là yếu tố dự báo quan trọng nhất.
2. **Thời gian sử dụng (tenure):** Khách hàng mới (có `tenure` thấp) có xu hướng rời bỏ dịch vụ nhiều hơn. Sau khoảng 20 tháng, tỷ lệ trung thành bắt đầu ổn định.
3. **Phương thức thanh toán (PaymentMethod):** Những khách hàng thanh toán bằng **Electronic check** có tỷ lệ Churn cao nhất trong các loại phương thức.
4. **Dịch vụ Internet (InternetService):** Khách hàng dùng **Fiber optic** (Cáp quang) rời mạng nhiều hơn DSL, có thể do chi phí cao hoặc sự cạnh tranh gay gắt ở phân khúc này.
5. **Hỗ trợ kỹ thuật (TechSupport):** Việc thiếu dịch vụ hỗ trợ kỹ thuật (`No TechSupport`) liên quan trực tiếp đến việc tăng tỷ lệ rời bỏ.

## 4. Chuẩn bị dữ liệu cho mô hình

Chia dữ liệu theo tỷ lệ 80% Train và 20% Test. Sử dụng `stratify` để đảm bảo tỷ lệ biến mục tiêu đồng nhất giữa hai tập.

In [4]:
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print('Train set:', X_train.shape)
print('Test set:', X_test.shape)


## 5. Huấn luyện và Đánh giá mô hình

Chúng ta huấn luyện đồng thời 3 mô hình và đánh giá qua Cross-Validation cũng như trên tập Test độc lập.

In [5]:
trained_models, metrics_df = train_and_evaluate_models(X_train, X_test, y_train, y_test)
display(metrics_df)
plot_model_metrics(metrics_df)


### 📊 Phân tích kết quả:
- Mô hình **Random Forest** thường đạt F1-score và Accuracy cao nhất nhờ khả năng xử lý tốt các quan hệ phi tuyến tính.
- Điểm **Recall** được ưu tiên cao (đạt khoảng 77-78%) để đảm bảo hệ thống không bỏ sót những khách hàng thực sự có nguy cơ rời mạng.

## 6. Ma trận nhầm lẫn (Confusion Matrix)

Biểu đồ này giúp ta nhìn rõ các lỗi của mô hình: dự đoán sai khách hàng ở lại thành rời đi (FP) hoặc ngược lại (FN).

In [6]:
for model_name, model in trained_models.items():
    plot_confusion_matrix(model, X_test, y_test, model_name)


## 7. Dự đoán thử nghiệm

Thực hiện dự đoán trên một mẫu khách hàng có nguy cơ cao để kiểm tra tính thực tiễn.

In [7]:
from src.data_processing import make_single_customer_example
from src.modeling import predict_churn_probability

best_model = trained_models[metrics_df.iloc[0]['model']]
customer = make_single_customer_example()
result = predict_churn_probability(best_model, customer)

print("Thông tin khách hàng mẫu (Nguy cơ cao):")
display(pd.DataFrame([customer]))
print("\nKết quả dự đoán từ mô hình:")
display(result)


## 8. Kết luận

Quy trình Machine Learning đã hoàn tất với độ tin cậy ổn định. Mô hình đã sẵn sàng để tích hợp vào ứng dụng Web phục vụ bộ phận Chăm sóc khách hàng.